# B.3 工具結果怎麼回到對話？


請求乘法123×45通過驗證後，外層程式真的算出5535。下一步不是再讓模型憑記憶猜答案，而是把這個返回值放回對話，讓它根據已取得的結果回答。前置是[請求驗證與本地執行](https://birdhackor.github.io/tiny-perceptron-vlm/B.2.html)：工具是確定的加法或乘法函數，模型負責提出請求與組織最後回答，兩者的紀錄需要分得清楚。

下面的trace是執行軌跡，也就是按順序保存「提出了什麼、工具回了什麼」。assistant角色代表模型這一側的請求，tool角色代表外層程式實際取得的結果。本例先人工固定request，方便把流程核對清楚；它沒有測試模型是否能自己選中multiply或填對數字。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.retrieval import call_tool

request = {"name": "multiply", "arguments": {"a": 123, "b": 45}}
result = call_tool(request)
trace = [
    {"role": "assistant", "tool_request": request},
    {"role": "tool", "result": result},
]
print("返回值", result)
print("將送回對話的兩筆紀錄", trace)
assert result == 5535

第一行應印5535，第二行第一筆含123與45，第二筆含result=5535。關鍵是result真的來自`call_tool`，不是自己寫一個期待答案塞進trace。`assert`再核對這個固定例子的乘法值。若後面接語言模型，還需把這兩筆Python紀錄轉成模型接受的對話文字或訊息格式，這個轉換稱為序列化；把轉換後的內容當作輸入，才能讓模型生成最終文字。本程式到建立紀錄為止，沒有假裝產生模型回答。

下圖從左到右展開同一流程。name+arguments是請求欄位；parse+validate是解析與驗證；add/multiply是允許的純函數，它們只依輸入數值算返回值；回填結果才是這節的tool紀錄。最後done表示任務完成信號，上限則避免循環一直延續，下一節會實作這兩種停止方式。箭頭表示資料處理順序，沒有任何一格會因模型說「完成了」就自動略過實際執行。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/tools.svg")))

回到最初問題，若模型把45誤寫成54，工具會正確算出6642，但最終仍答錯123×45。若請求正確、返回5535，模型卻最後寫成5355，就是閱讀或輸出階段的錯誤。因此評估不能只有「工具沒有報錯」，還要對照原問題、參數、返回值及最後回答。對簡單算術，可直接把返回值和真值比對；對較複雜任務，則需要明確的完成判準。

練習只把request的b改成46，並把固定例子的assert期待值改為5658。先預測tool紀錄與返回值會同步變5658，a仍是123，再執行核對。這能驗證資料真的由參數流到返回值；如果trace仍出現5535，就應檢查是否誤用舊結果或硬寫答案。
